# Mamba Monolith Notebook

Bu notebook, mevcut Mamba entegrasyonunun egitim, checkpoint ve inference yuzeyini tek dosyada toplar.

Kapsam:
- CSV -> normalize event -> session extraction
- vocab/codec olusturma
- Mamba model tanimi
- train / eval / checkpoint save-load
- checkpoint'ten inference ve materialization

Notlar:
- Bu notebook proje icindeki Python modullerini import etmez; cekirdek kod burada monolitik olarak bulunur.
- Varsayilan ayarlar GTX 1650 4 GB VRAM icin dusuktur.
- `mamba-ssm`, `torch`, `pandas`, `pyyaml` kurulu olmalidir.
- `MAMBA_MONOLITH_SMOKE=1` ile hizli smoke modu acilir.
- Bu dosya smoke modunda execute edilmis output alanlariyla saklanabilir; tam egitim icin smoke modunu kapatin.


## Core Contracts

Sabitler, veri siniflari ve temel dogrulama kurallari.


In [ ]:
from __future__ import annotations

# Bu hucre notebook'un tum monolitik cekirdegini tasir.
# Amac: proje icindeki daginik Mamba entegrasyonunu tek dosyada toplamak.
# Buradaki kod; veri okuma, tokenizasyon, model, egitim, checkpoint ve inference
# icin gereken cekirdek yuzeyi tek yerde tanimlar.

import copy
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Literal, Optional, Sequence, Tuple

import torch
import torch.nn as nn

# Dis bagimliliklar burada acikca fail-fast davranir.
# Notebook yarim yuklenip sonradan anlamsiz hata vermesin diye,
# eksik kutuphaneler hucrenin basinda net bir RuntimeError uretir.
try:
    import yaml
except Exception as exc:
    raise RuntimeError('pyyaml gerekli: pip install pyyaml') from exc

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError('pandas gerekli: pip install pandas') from exc

try:
    from mamba_ssm import Mamba  # type: ignore
except Exception:
    try:
        from mamba_ssm.modules.mamba_simple import Mamba  # type: ignore
    except Exception as exc:
        raise RuntimeError('mamba-ssm gerekli: pip install mamba-ssm') from exc

try:
    import mamba_ssm.modules.mamba_simple as _mamba_simple_mod  # type: ignore
    import mamba_ssm.ops.selective_scan_interface as _ssi  # type: ignore

    # Mamba wheel'i bazen sadece CUDA kernel ile gelir.
    # CPU tensor'lerinde referans uygulamaya dusen bir dispatch koyuyoruz.
    # Bu sayede notebook hem CPU smoke testlerinde hem de CUDA egitimde calisabilir.
    _orig_selective_scan_fn = _ssi.selective_scan_fn

    def _selective_scan_dispatch(
        u,
        delta,
        A,
        B,
        C,
        D=None,
        z=None,
        delta_bias=None,
        delta_softplus=False,
        return_last_state=False,
    ):
        if getattr(u, 'is_cuda', False):
            return _orig_selective_scan_fn(
                u,
                delta,
                A,
                B,
                C,
                D=D,
                z=z,
                delta_bias=delta_bias,
                delta_softplus=delta_softplus,
                return_last_state=return_last_state,
            )
        return _ssi.selective_scan_ref(
            u,
            delta,
            A,
            B,
            C,
            D=D,
            z=z,
            delta_bias=delta_bias,
            delta_softplus=delta_softplus,
            return_last_state=return_last_state,
        )

    _ssi.selective_scan_fn = _selective_scan_dispatch
    _mamba_simple_mod.selective_scan_fn = _selective_scan_dispatch
except Exception:
    pass

SUPPORTED_PROTOCOLS = {'http1', 'http2', 'websocket', 'quic'}
SUPPORTED_LABELS = {'benign', 'evasive', 'aggressive'}
SUPPORTED_EVENT_TYPES = {
    'connect_open',
    'tls_client_hello',
    'http_request',
    'http2_headers',
    'ws_open',
    'ws_frame',
    'quic_open',
    'quic_request',
    'idle_wait',
    'retry',
    'reset',
    'close',
}
PROTOCOL_EVENT_TYPES = {
    'http1': {'connect_open', 'tls_client_hello', 'http_request', 'idle_wait', 'retry', 'reset', 'close'},
    'http2': {'connect_open', 'tls_client_hello', 'http2_headers', 'idle_wait', 'retry', 'reset', 'close'},
    'websocket': {'connect_open', 'tls_client_hello', 'ws_open', 'ws_frame', 'idle_wait', 'retry', 'reset', 'close'},
    'quic': {'connect_open', 'quic_open', 'quic_request', 'idle_wait', 'retry', 'reset', 'close'},
}

PAD_TOKEN = '<pad>'
BOS_TOKEN = '<bos>'
EOS_TOKEN = '<eos>'
PROTO_PREFIX = 'PROTO:'
LABEL_PREFIX = 'LABEL:'
EVT_PREFIX = 'EVT:'
DELAY_PREFIX = 'DELAY:'
METHOD_PREFIX = 'METHOD:'
PATH_PREFIX = 'PATH:'
HEADER_PREFIX = 'HEADER:'
PAYLOAD_PREFIX = 'PAYLOAD:'
MUTATION_PREFIX = 'MUTATION:'


@dataclass(frozen=True)
class EventRecord:
    event_type: str
    delay_bucket: str = 'none'
    method: str = 'none'
    path_bucket: str = 'other'
    header_profile: str = 'none'
    payload_class: str = 'none'
    mutation_signature: str = 'none'


@dataclass(frozen=True)
class SessionRecord:
    session_id: str
    protocol: str
    label: str
    events: List[EventRecord]


@dataclass(frozen=True)
class SessionEvent:
    step_index: int
    event_type: str
    delay_ms: int = 0
    method: Optional[str] = None
    path: Optional[str] = None
    header_profile: Optional[str] = None
    payload_class: Optional[str] = None
    mutation_flags: List[str] = field(default_factory=list)
    expect_response_class: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class SessionPlan:
    session_id: str
    protocol: Literal['http1', 'http2', 'websocket', 'quic']
    label: Literal['benign', 'evasive', 'aggressive']
    events: List[SessionEvent]

    def to_dict(self) -> Dict[str, Any]:
        return {
            'session_id': self.session_id,
            'protocol': self.protocol,
            'label': self.label,
            'events': [ev.to_dict() for ev in self.events],
        }


@dataclass(frozen=True)
class TrafficPlan:
    plan_id: str
    model_name: str
    checkpoint_id: str
    seed: int
    session_plans: List[SessionPlan]

    def to_dict(self) -> Dict[str, Any]:
        return {
            'plan_id': self.plan_id,
            'model_name': self.model_name,
            'checkpoint_id': self.checkpoint_id,
            'seed': int(self.seed),
            'session_plans': [sp.to_dict() for sp in self.session_plans],
        }


@dataclass
class ValidationResult:
    valid: bool
    errors: List[str] = field(default_factory=list)
    warnings: List[str] = field(default_factory=list)


# Inference sonrasinda uretilen planlari burada tekrar dogruluyoruz.
# Sebep: model token sirasi uretse bile runtime semantigi acisindan
# protokol-event uyumu ve baslangic/bitis kurallari bozulmus olabilir.
def validate_plan(plan: TrafficPlan) -> ValidationResult:
    errors: List[str] = []
    warnings: List[str] = []
    if not str(plan.plan_id or '').strip():
        errors.append('plan_id is required')
    if not str(plan.model_name or '').strip():
        errors.append('model_name is required')
    if not plan.session_plans:
        errors.append('session_plans must be non-empty')
    for session_plan in plan.session_plans:
        if session_plan.protocol not in SUPPORTED_PROTOCOLS:
            errors.append(f'{session_plan.session_id}: unsupported protocol={session_plan.protocol}')
        if session_plan.label not in SUPPORTED_LABELS:
            errors.append(f'{session_plan.session_id}: unsupported label={session_plan.label}')
        if not session_plan.events:
            errors.append(f'{session_plan.session_id}: empty event sequence')
            continue
        if session_plan.events[0].event_type != 'connect_open':
            errors.append(f'{session_plan.session_id}: first event must be connect_open')
        if session_plan.events[-1].event_type not in {'close', 'reset'}:
            errors.append(f'{session_plan.session_id}: last event must be close|reset')
        allowed = PROTOCOL_EVENT_TYPES.get(session_plan.protocol, set())
        prev_idx = -1
        for ev in session_plan.events:
            if ev.step_index != prev_idx + 1:
                errors.append(f'{session_plan.session_id}: step index jump at {ev.step_index} (prev={prev_idx})')
            if ev.event_type not in SUPPORTED_EVENT_TYPES:
                errors.append(f'{session_plan.session_id}: unsupported event_type={ev.event_type}')
            if allowed and ev.event_type not in allowed:
                errors.append(f'{session_plan.session_id}: event_type={ev.event_type} invalid for protocol={session_plan.protocol}')
            if int(ev.delay_ms) < 0:
                errors.append(f'{session_plan.session_id}: negative delay_ms={ev.delay_ms}')
            if ev.event_type in {'http_request', 'http2_headers', 'quic_request'}:
                if not str(ev.method or '').strip():
                    errors.append(f'{session_plan.session_id}: request event missing method at step={ev.step_index}')
                if not str(ev.path or '').strip():
                    errors.append(f'{session_plan.session_id}: request event missing path at step={ev.step_index}')
            prev_idx = int(ev.step_index)
    if len(plan.session_plans) < 2:
        warnings.append('very small plan: session_plans < 2')
    return ValidationResult(valid=(len(errors) == 0), errors=errors, warnings=warnings)


@dataclass(frozen=True)
class NormalizedWireEvent:
    trace_id: str
    session_id: str
    event_index: int
    timestamp_rel_ms: int
    protocol: str
    event_type: str
    label: str
    src_role: str
    method: str
    path: str
    status_class: str
    payload_class: str
    header_profile: str
    mutation_flags: str
    tls_profile: str
    h2_profile: str
    quic_mode: str
    response_time_bucket: str
    retry_count: int
    close_reason: str



## CSV Normalization

CSV satirlarini normalize event dizilerine ve session kayitlarina ceviren katman.


In [ ]:

def _resolve_label(protocol_profile: str) -> str:
    prof = str(protocol_profile or '').strip()
    if prof.startswith('label='):
        return str(prof.split('=', 1)[1] or 'benign').strip().lower() or 'benign'
    return 'benign'


def _canonical_session_id(src_ip: str, sport: int, dst_ip: str, dport: int) -> str:
    a = (str(src_ip), int(sport))
    b = (str(dst_ip), int(dport))
    left, right = (a, b) if a <= b else (b, a)
    return f'{left[0]}:{left[1]}<->{right[0]}:{right[1]}'


def _canon(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(name or '').strip().lower())


def _trace_id_for_file(csv_path: str) -> str:
    p = Path(csv_path)
    h = hashlib.sha1(str(p.resolve()).encode('utf-8', errors='replace')).hexdigest()[:12]
    return f'{p.stem}-{h}'


def _to_int(v, default: int = 0) -> int:
    try:
        if v is None:
            return int(default)
        s = str(v).strip()
        if not s:
            return int(default)
        if '.' in s:
            return int(float(s))
        return int(s)
    except Exception:
        return int(default)


def _payload_class_bytes(total_bytes: int) -> str:
    n = int(total_bytes or 0)
    if n <= 0:
        return 'empty'
    if n <= 128:
        return 'tiny'
    if n <= 1024:
        return 'small'
    if n <= 8192:
        return 'medium'
    return 'large'


def _map_label(raw_label: str, default_label: str = 'benign') -> str:
    s = str(raw_label or '').strip().lower()
    if not s:
        return str(default_label or 'benign')
    if s in {'benign', 'normal'} or 'benign' in s:
        return 'benign'
    evasive_keywords = ('slowloris', 'slowhttptest', 'brute', 'infiltration', 'web attack', 'xss', 'sql', 'heartbleed', 'bot')
    if any(k in s for k in evasive_keywords):
        return 'evasive'
    return 'aggressive'


def _protocol_from_row(proto_raw, src_port: int, dst_port: int) -> str:
    p = str(proto_raw or '').strip().lower()
    if not p:
        if src_port == 443 or dst_port == 443:
            return 'http2'
        return 'http1'
    if p.isdigit():
        n = int(p)
        if n == 17:
            return 'quic' if (src_port == 443 or dst_port == 443) else 'http1'
        if n == 6:
            return 'http2' if (src_port == 443 or dst_port == 443) else 'http1'
    if p in {'udp'}:
        return 'quic' if (src_port == 443 or dst_port == 443) else 'http1'
    if p in {'tcp'}:
        return 'http2' if (src_port == 443 or dst_port == 443) else 'http1'
    if 'quic' in p:
        return 'quic'
    if 'http2' in p or p == 'h2':
        return 'http2'
    return 'http1'


def _build_colmap(columns: Iterable[str]) -> Dict[str, str]:
    out: Dict[str, str] = {}
    for c in list(columns):
        cc = _canon(c)
        if cc and cc not in out:
            out[cc] = str(c)
    return out


def _first_non_empty(row: dict, colmap: Dict[str, str], keys: List[str], default: str = '') -> str:
    for k in keys:
        real = colmap.get(_canon(k))
        if not real:
            continue
        v = row.get(real)
        s = str(v).strip() if v is not None else ''
        if s:
            return s
    return str(default or '')


def iter_csv_normalized_events(
    csv_path: str,
    protocol_profile: str = '',
    *,
    default_label: str = 'benign',
    max_rows: int = 0,
    chunksize: int = 200000,
) -> Iterator[NormalizedWireEvent]:
    # CSV satirlarini dogrudan modelin tukecegi kaba event sekansina indirgeriz.
    # Burada hedef tam paket seviyesinde rekonstruksiyon degil;
    # session -> event transition mantigini ogrenebilecegimiz tutarli bir temsil uretmektir.
    file_path = str(csv_path or '').strip()
    if not file_path:
        return
    p = Path(file_path)
    if not p.exists():
        return

    trace_id = _trace_id_for_file(file_path)
    fallback_label = _resolve_label(protocol_profile) or str(default_label or 'benign')
    session_next_idx: Dict[str, int] = {}
    row_count = 0
    first_ts = None

    reader = pd.read_csv(file_path, low_memory=False, chunksize=max(1, int(chunksize or 200000)))
    for chunk in reader:
        colmap = _build_colmap(list(chunk.columns))
        for row in chunk.to_dict(orient='records'):
            row_count += 1
            if int(max_rows or 0) > 0 and row_count > int(max_rows):
                return

            src_ip = _first_non_empty(row, colmap, ['source ip', 'src ip'], default='0.0.0.0')
            dst_ip = _first_non_empty(row, colmap, ['destination ip', 'dst ip'], default='0.0.0.0')
            src_port = _to_int(_first_non_empty(row, colmap, ['source port', 'src port'], default='0'), default=0)
            dst_port = _to_int(_first_non_empty(row, colmap, ['destination port', 'dst port'], default='0'), default=0)
            proto_raw = _first_non_empty(row, colmap, ['protocol'], default='')
            protocol = _protocol_from_row(proto_raw, src_port, dst_port)

            row_label = _first_non_empty(row, colmap, ['label', 'attack', 'class'], default=fallback_label)
            label = _map_label(row_label, default_label=fallback_label)

            flow_id = _first_non_empty(row, colmap, ['flow id', 'flowid'], default='')
            if flow_id:
                session_id = f'flow:{flow_id}'
            else:
                if src_ip == '0.0.0.0' and dst_ip == '0.0.0.0' and src_port == 0:
                    session_id = f'row:{trace_id}:{row_count}'
                else:
                    session_id = _canonical_session_id(src_ip, src_port, dst_ip, dst_port)

            ts_raw = _first_non_empty(row, colmap, ['timestamp', 'time'], default='')
            ts_val = None
            if ts_raw:
                try:
                    ts_obj = pd.to_datetime(ts_raw, errors='coerce')
                    if hasattr(ts_obj, 'value'):
                        ts_val = float(ts_obj.value) / 1_000_000_000.0
                except Exception:
                    ts_val = None
            if ts_val is None:
                ts_val = float(row_count)
            if first_ts is None:
                first_ts = float(ts_val)
            base_rel_ms = int(max(0.0, (float(ts_val) - float(first_ts)) * 1000.0))

            syn_cnt = _to_int(_first_non_empty(row, colmap, ['syn flag cnt', 'synflagcnt'], default='0'), default=0)
            rst_cnt = _to_int(_first_non_empty(row, colmap, ['rst flag cnt', 'rstflagcnt'], default='0'), default=0)
            fin_cnt = _to_int(_first_non_empty(row, colmap, ['fin flag cnt', 'finflagcnt'], default='0'), default=0)
            fwd_bytes = _to_int(_first_non_empty(row, colmap, ['total length of fwd packets', 'totallengthoffwdpackets', 'fwd packet length total', 'fwdpacketlengthtotal'], default='0'), default=0)
            bwd_bytes = _to_int(_first_non_empty(row, colmap, ['total length of bwd packets', 'totallengthofbwdpackets', 'bwd packet length total', 'bwdpacketlengthtotal'], default='0'), default=0)
            payload_class = _payload_class_bytes(fwd_bytes + bwd_bytes)

            if protocol == 'quic':
                req_event_type = 'quic_request'
                header_profile = 'quic_csv'
                tls_profile = 'none'
                h2_profile = 'none'
                quic_mode = 'csv_inferred'
            elif protocol == 'http2':
                req_event_type = 'http2_headers'
                header_profile = 'h2_csv'
                tls_profile = 'csv_inferred'
                h2_profile = 'csv_inferred'
                quic_mode = 'none'
            else:
                req_event_type = 'http_request'
                header_profile = 'http_csv'
                tls_profile = 'none'
                h2_profile = 'none'
                quic_mode = 'none'

            mutation_flags = 'csv_attack_label' if label != 'benign' else ''
            sequence = ['connect_open', req_event_type, 'close']
            if rst_cnt > 0:
                sequence[-1] = 'reset'
            elif fin_cnt > 0:
                sequence[-1] = 'close'
            elif syn_cnt > 0:
                sequence[-1] = 'close'

            for offset, ev_type in enumerate(sequence):
                idx = int(session_next_idx.get(session_id, 0))
                session_next_idx[session_id] = idx + 1
                method = 'GET' if ev_type in {'http_request', 'http2_headers', 'quic_request'} else ''
                path = '/' if method else ''
                close_reason = 'rst' if ev_type == 'reset' else ('fin' if ev_type == 'close' else '')
                yield NormalizedWireEvent(
                    trace_id=trace_id,
                    session_id=session_id,
                    event_index=idx,
                    timestamp_rel_ms=int(base_rel_ms + offset),
                    protocol=protocol,
                    event_type=ev_type,
                    label=label,
                    src_role='client',
                    method=method,
                    path=path,
                    status_class='',
                    payload_class=payload_class,
                    header_profile=header_profile,
                    mutation_flags=mutation_flags,
                    tls_profile=tls_profile,
                    h2_profile=h2_profile,
                    quic_mode=quic_mode,
                    response_time_bucket='unknown',
                    retry_count=0,
                    close_reason=close_reason,
                )


def _delay_bucket(delta_ms: int) -> str:
    n = max(0, int(delta_ms or 0))
    if n <= 5:
        return '0_5ms'
    if n <= 20:
        return '6_20ms'
    if n <= 100:
        return '21_100ms'
    if n <= 500:
        return '101_500ms'
    return '500ms_plus'


def _path_bucket(path: str) -> str:
    p = str(path or '').strip().lower()
    if not p or p == '/':
        return 'root'
    if p.startswith('/api'):
        return 'api'
    if p.startswith('/ws'):
        return 'ws'
    if p.startswith('/static'):
        return 'static'
    if p.startswith('/login') or p.startswith('/auth'):
        return 'auth'
    if p.startswith('/search'):
        return 'search'
    if p.startswith('/image') or p.startswith('/img') or p.startswith('/media'):
        return 'media'
    return 'other'


def _mutation_signature(raw: str) -> str:
    parts = sorted({str(x).strip() for x in str(raw or '').split(',') if str(x).strip()})
    if not parts:
        return 'none'
    return '|'.join(parts)


def infer_label_from_path(path: Path) -> str:
    labels = {'benign', 'evasive', 'aggressive'}
    for part in reversed(path.parts[:-1]):
        key = str(part).strip().lower()
        if key in labels:
            return key
    return ''


def collect_csv_sources(*, csv_dir: str = '', csv_glob: str = '*.csv', csv_files: Sequence[str] | None = None) -> List[Tuple[str, Path]]:
    out: List[Tuple[str, Path]] = []
    seen: set[str] = set()
    for raw in list(csv_files or []):
        p = Path(str(raw or '').strip())
        if not p.exists() or not p.is_file():
            continue
        rp = str(p.resolve())
        if rp in seen:
            continue
        seen.add(rp)
        out.append((infer_label_from_path(p), p))
    root = Path(str(csv_dir or '').strip()) if str(csv_dir or '').strip() else None
    if root is not None and root.exists():
        for p in sorted(root.rglob(str(csv_glob or '*.csv'))):
            if not p.is_file():
                continue
            rp = str(p.resolve())
            if rp in seen:
                continue
            seen.add(rp)
            out.append((infer_label_from_path(p), p))
    return out


def load_sessions_from_csv_sources(
    sources: Sequence[Tuple[str, Path]],
    *,
    max_sessions: int = 0,
    max_rows_per_file: int = 0,
    csv_chunksize: int = 50000,
) -> Tuple[List[SessionRecord], int, int]:
    events_by_sid: Dict[str, List[Tuple[int, EventRecord]]] = {}
    meta_by_sid: Dict[str, Tuple[str, str]] = {}
    prev_ts_by_sid: Dict[str, int] = {}
    sid_order: List[str] = []
    rows_seen = 0
    files_seen = 0
    max_sess = max(0, int(max_sessions or 0))

    def _accept_sid(sid: str) -> bool:
        if sid in events_by_sid:
            return True
        if max_sess > 0 and len(events_by_sid) >= max_sess:
            return False
        events_by_sid[sid] = []
        sid_order.append(sid)
        return True

    for label_hint, csv_path in list(sources or []):
        files_seen += 1
        label_for_profile = str(label_hint or '').strip().lower()
        profile = f'label={label_for_profile}' if label_for_profile else ''
        for ev in iter_csv_normalized_events(
            str(csv_path),
            protocol_profile=profile,
            default_label=label_for_profile or 'benign',
            max_rows=int(max_rows_per_file or 0),
            chunksize=max(1, int(csv_chunksize or 50000)),
        ):
            rows_seen += 1
            sid_raw = str(getattr(ev, 'session_id', '') or '').strip()
            if not sid_raw:
                continue
            trace_id = str(getattr(ev, 'trace_id', '') or '').strip()
            sid = f'{trace_id}::{sid_raw}' if trace_id else sid_raw
            if not _accept_sid(sid):
                continue
            if sid not in meta_by_sid:
                meta_by_sid[sid] = (
                    str(getattr(ev, 'protocol', 'http1') or 'http1'),
                    str(getattr(ev, 'label', 'benign') or 'benign'),
                )
            ev_type = str(getattr(ev, 'event_type', '') or '').strip()
            if not ev_type:
                continue
            try:
                eidx = int(getattr(ev, 'event_index', 0) or 0)
            except Exception:
                eidx = 0
            ts_rel_ms = int(getattr(ev, 'timestamp_rel_ms', 0) or 0)
            prev_ts = int(prev_ts_by_sid.get(sid, ts_rel_ms))
            delay_bucket = _delay_bucket(ts_rel_ms - prev_ts if sid in prev_ts_by_sid else ts_rel_ms)
            prev_ts_by_sid[sid] = ts_rel_ms
            event = EventRecord(
                event_type=ev_type,
                delay_bucket=delay_bucket,
                method=str(getattr(ev, 'method', '') or '').strip().upper() or 'none',
                path_bucket=_path_bucket(str(getattr(ev, 'path', '') or '')),
                header_profile=str(getattr(ev, 'header_profile', '') or '').strip() or 'none',
                payload_class=str(getattr(ev, 'payload_class', '') or '').strip() or 'none',
                mutation_signature=_mutation_signature(str(getattr(ev, 'mutation_flags', '') or '')),
            )
            events_by_sid[sid].append((eidx, event))

    sessions: List[SessionRecord] = []
    for sid in sid_order:
        pairs = sorted(events_by_sid.get(sid, []), key=lambda x: int(x[0]))
        events = [ev for _i, ev in pairs if ev]
        if not events:
            continue
        protocol, label = meta_by_sid.get(sid, ('http1', 'benign'))
        sessions.append(SessionRecord(session_id=sid, protocol=protocol, label=label, events=events))
    return sessions, int(rows_seen), int(files_seen)



## Codec And Model

TokenCodec, dataset, Mamba model ve materialization yardimcilari.


In [ ]:

class TokenCodec:
    # TokenCodec, session verisini otoregresif token dizisine cevirir.
    # Protokol ve label condition token'lari basa eklenir.
    # Her event daha sonra cok alanli olarak serialize edilir:
    # event_type, delay_bucket, method, path_bucket, header_profile,
    # payload_class ve mutation_signature.
    def __init__(self, vocab: Sequence[str]) -> None:
        vv = [str(x) for x in vocab]
        if not vv or vv[0] != PAD_TOKEN or vv[1] != BOS_TOKEN or vv[2] != EOS_TOKEN:
            raise ValueError('invalid vocabulary header')
        self.vocab: List[str] = vv
        self.token_to_id: Dict[str, int] = {t: i for i, t in enumerate(self.vocab)}
        self.pad_id = self.token_to_id[PAD_TOKEN]
        self.bos_id = self.token_to_id[BOS_TOKEN]
        self.eos_id = self.token_to_id[EOS_TOKEN]
        self.event_token_ids = [i for i, t in enumerate(self.vocab) if t.startswith(EVT_PREFIX)]
        self.delay_token_ids = [i for i, t in enumerate(self.vocab) if t.startswith(DELAY_PREFIX)]
        self.method_token_ids = [i for i, t in enumerate(self.vocab) if t.startswith(METHOD_PREFIX)]
        self.path_token_ids = [i for i, t in enumerate(self.vocab) if t.startswith(PATH_PREFIX)]
        self.header_token_ids = [i for i, t in enumerate(self.vocab) if t.startswith(HEADER_PREFIX)]
        self.payload_token_ids = [i for i, t in enumerate(self.vocab) if t.startswith(PAYLOAD_PREFIX)]
        self.mutation_token_ids = [i for i, t in enumerate(self.vocab) if t.startswith(MUTATION_PREFIX)]
        self._protocol_event_token_ids: Dict[str, List[int]] = {}
        for protocol, event_types in PROTOCOL_EVENT_TYPES.items():
            ids: List[int] = []
            for event_type in sorted(event_types):
                tok = f'{EVT_PREFIX}{event_type}'
                token_id = self.token_to_id.get(tok)
                if token_id is not None:
                    ids.append(int(token_id))
            self._protocol_event_token_ids[str(protocol)] = ids

    @classmethod
    def build(cls, sessions: Sequence[SessionRecord]) -> 'TokenCodec':
        protos = sorted({f'{PROTO_PREFIX}{s.protocol}' for s in sessions if s.protocol})
        labels = sorted({f'{LABEL_PREFIX}{s.label}' for s in sessions if s.label})
        event_types = sorted({f'{EVT_PREFIX}{ev.event_type}' for s in sessions for ev in list(s.events or []) if getattr(ev, 'event_type', '')})
        delays = sorted({f'{DELAY_PREFIX}{ev.delay_bucket or "none"}' for s in sessions for ev in list(s.events or [])})
        methods = sorted({f'{METHOD_PREFIX}{(ev.method or "none").lower()}' for s in sessions for ev in list(s.events or [])})
        paths = sorted({f'{PATH_PREFIX}{ev.path_bucket or "other"}' for s in sessions for ev in list(s.events or [])})
        headers = sorted({f'{HEADER_PREFIX}{ev.header_profile or "none"}' for s in sessions for ev in list(s.events or [])})
        payloads = sorted({f'{PAYLOAD_PREFIX}{ev.payload_class or "none"}' for s in sessions for ev in list(s.events or [])})
        mutations = sorted({f'{MUTATION_PREFIX}{ev.mutation_signature or "none"}' for s in sessions for ev in list(s.events or [])})
        vocab = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN] + protos + labels + event_types + delays + methods + paths + headers + payloads + mutations
        return cls(vocab=vocab)

    def encode_token(self, token: str) -> int:
        return int(self.token_to_id[token])

    def _encode_condition(self, prefix: str, value: str) -> int:
        direct = f'{prefix}{value}'
        if direct in self.token_to_id:
            return int(self.token_to_id[direct])
        raise KeyError(f'unknown conditioned token: {direct}')

    def decode_token(self, token_id: int) -> str:
        return str(self.vocab[int(token_id)])

    def encode_sequence(self, protocol: str, label: str, events: Sequence[EventRecord | str], max_len: int) -> List[int]:
        seq = [self.bos_id, self._encode_condition(PROTO_PREFIX, protocol), self._encode_condition(LABEL_PREFIX, label)]
        for ev in list(events or []):
            event = ev if isinstance(ev, EventRecord) else EventRecord(event_type=str(ev))
            field_tokens = [
                f'{EVT_PREFIX}{str(event.event_type)}',
                f'{DELAY_PREFIX}{str(event.delay_bucket or "none")}',
                f'{METHOD_PREFIX}{str(event.method or "none").lower()}',
                f'{PATH_PREFIX}{str(event.path_bucket or "other")}',
                f'{HEADER_PREFIX}{str(event.header_profile or "none")}',
                f'{PAYLOAD_PREFIX}{str(event.payload_class or "none")}',
                f'{MUTATION_PREFIX}{str(event.mutation_signature or "none")}',
            ]
            for tok in field_tokens:
                if tok not in self.token_to_id:
                    raise KeyError(f'unknown field token: {tok}')
                seq.append(self.encode_token(tok))
            if len(seq) >= max_len - 1:
                break
        seq.append(self.eos_id)
        return seq[:max_len]

    def decode_event_token(self, token_id: int) -> str | None:
        tok = self.decode_token(token_id)
        if tok.startswith(EVT_PREFIX):
            return tok[len(EVT_PREFIX):]
        return None

    def allowed_event_token_ids(self, protocol: str) -> List[int]:
        key = str(protocol or '').strip().lower()
        ids = list(self._protocol_event_token_ids.get(key, []))
        if not ids:
            raise KeyError(f'no event vocabulary coverage for protocol={protocol}')
        return ids

    def allowed_field_token_ids(self, prefix: str) -> List[int]:
        mapping = {
            DELAY_PREFIX: self.delay_token_ids,
            METHOD_PREFIX: self.method_token_ids,
            PATH_PREFIX: self.path_token_ids,
            HEADER_PREFIX: self.header_token_ids,
            PAYLOAD_PREFIX: self.payload_token_ids,
            MUTATION_PREFIX: self.mutation_token_ids,
        }
        ids = list(mapping.get(prefix, []))
        if not ids:
            raise KeyError(f'no tokens for prefix={prefix}')
        return ids

    def token_id_for(self, prefix: str, value: str) -> int:
        tok = f'{prefix}{value}'
        if tok not in self.token_to_id:
            raise KeyError(f'unknown token: {tok}')
        return int(self.token_to_id[tok])


def _default_event_record(event_type: str) -> EventRecord:
    request_like = event_type in {'http_request', 'http2_headers', 'quic_request'}
    ws_like = event_type in {'ws_open', 'ws_frame'}
    return EventRecord(
        event_type=str(event_type),
        delay_bucket='6_20ms',
        method='get' if request_like else 'none',
        path_bucket='ws' if ws_like else ('api' if request_like else 'other'),
        header_profile='modern_browser' if request_like else 'none',
        payload_class='small' if request_like else 'none',
        mutation_signature='none',
    )


def _allowed_ids_for_event_field(codec: TokenCodec, prefix: str, protocol: str, event_type: str) -> List[int]:
    request_like = event_type in {'http_request', 'http2_headers', 'quic_request'}
    ws_like = event_type in {'ws_open', 'ws_frame'}
    if prefix == DELAY_PREFIX:
        return codec.allowed_field_token_ids(prefix)
    if prefix == METHOD_PREFIX:
        if request_like:
            return [i for i in codec.method_token_ids if codec.decode_token(i) != f'{METHOD_PREFIX}none']
        return [codec.token_id_for(METHOD_PREFIX, 'none')]
    if prefix == PATH_PREFIX:
        if request_like:
            return codec.path_token_ids
        if ws_like:
            return [codec.token_id_for(PATH_PREFIX, 'ws')]
        return [codec.token_id_for(PATH_PREFIX, 'other')]
    if prefix == HEADER_PREFIX:
        if request_like:
            return codec.header_token_ids
        return [codec.token_id_for(HEADER_PREFIX, 'none')]
    if prefix == PAYLOAD_PREFIX:
        if request_like:
            return codec.payload_token_ids
        return [codec.token_id_for(PAYLOAD_PREFIX, 'none')]
    if prefix == MUTATION_PREFIX:
        return codec.mutation_token_ids
    raise KeyError(f'unsupported prefix={prefix}')


class SequenceDataset(torch.utils.data.Dataset):
    # Dataset, degisken uzunluktaki token dizilerini pad ederek sabit max_len
    # tensore cevirir. Model next-token objective ile egitildigi icin
    # tek ihtiyacimiz input_ids tensorudur.
    def __init__(self, sessions: Sequence[SessionRecord], codec: TokenCodec, max_len: int) -> None:
        self.samples: List[List[int]] = []
        for session in list(sessions or []):
            try:
                encoded = codec.encode_sequence(session.protocol, session.label, session.events, int(max_len))
            except KeyError as exc:
                raise ValueError(f'session {session.session_id} is incompatible with codec: {exc}') from exc
            self.samples.append(encoded)
        self.max_len = int(max_len)
        self.pad_id = int(codec.pad_id)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        seq = list(self.samples[int(idx)])
        if len(seq) < self.max_len:
            seq.extend([self.pad_id] * (self.max_len - len(seq)))
        else:
            seq = seq[: self.max_len]
        return {'input_ids': torch.tensor(seq, dtype=torch.long)}


class ResidualMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mixer = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mixer(self.norm(x))


class MambaNextTokenModel(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int = 512, num_layers: int = 12, d_state: int = 16, d_conv: int = 4, expand: int = 2) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            ResidualMambaBlock(d_model=hidden_size, d_state=d_state, d_conv=d_conv, expand=expand)
            for _ in range(int(num_layers))
        ])
        self.norm_f = nn.LayerNorm(hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)
        return self.head(x)


def build_model_from_config(vocab_size: int, config: Dict[str, Any]) -> MambaNextTokenModel:
    model_cfg = dict((config or {}).get('model', {}) or {})
    return MambaNextTokenModel(
        vocab_size=int(vocab_size),
        hidden_size=int(model_cfg.get('hidden_size', 512) or 512),
        num_layers=int(model_cfg.get('num_layers', 12) or 12),
        d_state=int(model_cfg.get('d_state', 16) or 16),
        d_conv=int(model_cfg.get('d_conv', 4) or 4),
        expand=int(model_cfg.get('expand', 2) or 2),
    )


def training_device(raw: str) -> torch.device:
    req = str(raw or 'cpu').strip().lower()
    if req in {'cuda', 'gpu'}:
        if not torch.cuda.is_available():
            raise RuntimeError('cuda/gpu requested but not available')
        return torch.device('cuda')
    return torch.device('cpu')


def seed_everything(seed: int) -> None:
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


@torch.no_grad()
def evaluate_model(model: MambaNextTokenModel, loader: torch.utils.data.DataLoader, *, pad_id: int, device: torch.device) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    total_hit = 0
    total_pred = 0
    ce = nn.CrossEntropyLoss(ignore_index=int(pad_id), reduction='sum')
    for batch in loader:
        ids = batch['input_ids'].to(device)
        logits = model(ids)
        target = ids[:, 1:].contiguous()
        pred = logits[:, :-1, :].contiguous()
        loss = ce(pred.view(-1, pred.size(-1)), target.view(-1))
        total_loss += float(loss.item())
        mask = target.ne(int(pad_id))
        hit = pred.argmax(dim=-1).eq(target) & mask
        total_hit += int(hit.sum().item())
        total_pred += int(mask.sum().item())
        total_tokens += int(mask.sum().item())
    avg_nll = total_loss / float(max(1, total_tokens))
    ppl = float(math.exp(min(20.0, avg_nll)))
    acc = float(total_hit) / float(max(1, total_pred))
    return {
        'next_event_top1_accuracy': float(acc),
        'transition_perplexity': float(ppl),
        'transitions_eval': int(total_pred),
    }


def run_epoch(*, model: nn.Module, loader: torch.utils.data.DataLoader, optimizer: torch.optim.Optimizer, device: torch.device, pad_id: int, epoch: int, log_interval: int) -> Dict[str, float]:
    # Bu egitim dongusu bilincli olarak sade tutuldu.
    # Mixed precision, grad accumulation veya checkpointing eklemedik;
    # notebook'un okunabilirligini koruyup davranisi acik tutuyoruz.
    model.train()
    ce = nn.CrossEntropyLoss(ignore_index=int(pad_id), reduction='sum')
    total_loss = 0.0
    total_tokens = 0
    for step, batch in enumerate(loader, start=1):
        ids = batch['input_ids'].to(device)
        logits = model(ids)
        target = ids[:, 1:].contiguous()
        pred = logits[:, :-1, :].contiguous()
        loss = ce(pred.view(-1, pred.size(-1)), target.view(-1))
        token_count = int(target.ne(int(pad_id)).sum().item())
        norm_loss = loss / float(max(1, token_count))
        optimizer.zero_grad(set_to_none=True)
        norm_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += float(loss.item())
        total_tokens += token_count
        if int(log_interval or 0) > 0 and (step % int(log_interval)) == 0:
            step_nll = float(total_loss / float(max(1, total_tokens)))
            print(json.dumps({'epoch': int(epoch), 'step': int(step), 'steps_total': int(len(loader)), 'train_nll_running': float(step_nll), 'tokens_seen': int(total_tokens)}, sort_keys=True), flush=True)
    return {'nll': float(total_loss / float(max(1, total_tokens)))}


def save_checkpoint(path: str, *, model: MambaNextTokenModel, codec: TokenCodec, config: Dict[str, Any], dataset_meta: Dict[str, Any], metrics: Dict[str, Any], optimizer: Optional[torch.optim.Optimizer] = None, train_state: Optional[Dict[str, Any]] = None) -> None:
    # Resume training icin sadece model agirliklari yetmez.
    # Optimizer state'i ve tamamlanan epoch sayisini da kaydediyoruz.
    payload = {
        'format': 'mamba2_torch_v1',
        'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'model_name': str((config or {}).get('model', {}).get('name', 'mamba2-small')),
        'model_contract': 'multi_field_transition_lm_v2',
        'runtime_materialization': 'token_conditioned_v2',
        'config': dict(config or {}),
        'vocab': list(codec.vocab),
        'dataset': dict(dataset_meta or {}),
        'metrics': dict(metrics or {}),
        'state_dict': model.state_dict(),
        'optimizer_state_dict': (optimizer.state_dict() if optimizer is not None else None),
        'train_state': dict(train_state or {}),
    }
    torch.save(payload, str(path))


def load_checkpoint(path: str, *, map_location: str | torch.device = 'cpu') -> Dict[str, Any]:
    ck = torch.load(str(path), map_location=map_location)
    if not isinstance(ck, dict):
        raise RuntimeError('invalid checkpoint payload')
    if str(ck.get('format', '')) != 'mamba2_torch_v1':
        raise RuntimeError('checkpoint is not mamba2_torch_v1')
    return ck


def build_model_and_codec_from_checkpoint(checkpoint_path: str, *, device: str = 'cpu') -> Tuple[MambaNextTokenModel, TokenCodec, Dict[str, Any]]:
    map_loc = 'cpu' if str(device).lower() == 'cpu' else str(device)
    ck = load_checkpoint(checkpoint_path, map_location=map_loc)
    codec = TokenCodec(vocab=list(ck['vocab']))
    model = build_model_from_config(vocab_size=len(codec.vocab), config=dict(ck.get('config', {}) or {}))
    model.load_state_dict(dict(ck.get('state_dict', {}) or {}), strict=True)
    model.to(torch.device(device))
    model.eval()
    return model, codec, ck


@torch.no_grad()
def sample_event_types(model: MambaNextTokenModel, codec: TokenCodec, *, protocol: str, label: str, max_events: int, seed: int, device: str = 'cpu', temperature: float = 0.8, top_k: int = 20) -> List[EventRecord]:
    gen = torch.Generator(device='cpu')
    gen.manual_seed(int(seed))
    ids = codec.encode_sequence(protocol=protocol, label=label, events=[], max_len=3)
    out: List[EventRecord] = []
    model.eval()
    field_order = [EVT_PREFIX, DELAY_PREFIX, METHOD_PREFIX, PATH_PREFIX, HEADER_PREFIX, PAYLOAD_PREFIX, MUTATION_PREFIX]

    def _sample_from_allowed(allowed_ids: List[int]) -> int:
        x = torch.tensor([ids], dtype=torch.long, device=torch.device(device))
        logits = model(x)[:, -1, :].squeeze(0)
        mask = torch.full_like(logits, float('-inf'))
        mask[allowed_ids] = 0.0
        masked_logits = logits + mask
        temp = max(1e-4, float(temperature))
        masked_logits = masked_logits / temp
        if int(top_k) > 0 and int(top_k) < int(len(allowed_ids)):
            vals, idxs = torch.topk(masked_logits, k=int(top_k))
            probs = torch.softmax(vals, dim=-1)
            pick_local = int(torch.multinomial(probs, num_samples=1, generator=gen).item())
            return int(idxs[pick_local].item())
        probs = torch.softmax(masked_logits, dim=-1)
        return int(torch.multinomial(probs, num_samples=1, generator=gen).item())

    for _ in range(max(1, int(max_events))):
        next_id = _sample_from_allowed(list(codec.allowed_event_token_ids(protocol)) + [int(codec.eos_id)])
        if next_id == int(codec.eos_id):
            break
        ids.append(next_id)
        event_type = codec.decode_event_token(next_id)
        if not event_type:
            break
        event = _default_event_record(event_type)
        for prefix in field_order[1:]:
            field_id = _sample_from_allowed(_allowed_ids_for_event_field(codec, prefix, protocol, event_type))
            ids.append(field_id)
            token = codec.decode_token(field_id)
            value = token.split(':', 1)[1] if ':' in token else ''
            if prefix == DELAY_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=value or 'none', method=event.method, path_bucket=event.path_bucket, header_profile=event.header_profile, payload_class=event.payload_class, mutation_signature=event.mutation_signature)
            elif prefix == METHOD_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=value or 'none', path_bucket=event.path_bucket, header_profile=event.header_profile, payload_class=event.payload_class, mutation_signature=event.mutation_signature)
            elif prefix == PATH_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=event.method, path_bucket=value or 'other', header_profile=event.header_profile, payload_class=event.payload_class, mutation_signature=event.mutation_signature)
            elif prefix == HEADER_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=event.method, path_bucket=event.path_bucket, header_profile=value or 'none', payload_class=event.payload_class, mutation_signature=event.mutation_signature)
            elif prefix == PAYLOAD_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=event.method, path_bucket=event.path_bucket, header_profile=event.header_profile, payload_class=value or 'none', mutation_signature=event.mutation_signature)
            elif prefix == MUTATION_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=event.method, path_bucket=event.path_bucket, header_profile=event.header_profile, payload_class=event.payload_class, mutation_signature=value or 'none')
        out.append(event)
    return out


def materialize_session_events(event_types: Sequence[EventRecord], *, protocol: str, label: str, session_len: int, seed: int) -> List[Dict[str, Any]]:
    rng = random.Random(int(seed))
    events: List[Dict[str, Any]] = []
    allowed = PROTOCOL_EVENT_TYPES.get(str(protocol), set())
    if not allowed:
        raise ValueError(f'unsupported protocol for materialization: {protocol}')
    seq = [ev for ev in list(event_types or []) if getattr(ev, 'event_type', '')]
    invalid = [ev.event_type for ev in seq if ev.event_type not in allowed]
    if invalid:
        raise ValueError(f'event types incompatible with protocol={protocol}: {invalid}')
    if not seq or seq[0].event_type != 'connect_open':
        seq = [_default_event_record('connect_open')] + seq
    if len(seq) > int(session_len):
        seq = seq[: int(session_len)]
    if seq[-1].event_type != 'close':
        if len(seq) < int(session_len):
            seq.append(_default_event_record('close'))
        else:
            seq[-1] = _default_event_record('close')
    path_map = {'root': '/', 'api': '/api/v1/resource', 'ws': '/ws', 'static': '/static/app.js', 'auth': '/login', 'search': '/search', 'media': '/media/item', 'other': '/resource'}
    for i, ev in enumerate(seq):
        method = None
        path = None
        if ev.event_type in {'http_request', 'http2_headers', 'quic_request'}:
            method = str(ev.method or 'GET').upper()
            path = path_map.get(str(ev.path_bucket or 'other'), '/resource')
        elif ev.event_type in {'ws_open', 'ws_frame'}:
            path = '/ws'
        mutation_flags = []
        if str(ev.mutation_signature or 'none') != 'none':
            mutation_flags = [x for x in str(ev.mutation_signature).split('|') if x]
        if not mutation_flags and label in {'evasive', 'aggressive'} and rng.random() < 0.15:
            mutation_flags.append('header_shuffle')
        delay_ms_map = {'none': 0, '0_5ms': rng.randint(0, 5), '6_20ms': rng.randint(6, 20), '21_100ms': rng.randint(21, 100), '101_500ms': rng.randint(101, 500), '500ms_plus': rng.randint(501, 1000)}
        events.append({
            'step_index': int(i),
            'event_type': str(ev.event_type),
            'delay_ms': int(delay_ms_map.get(str(ev.delay_bucket or 'none'), rng.randint(1, 80))),
            'method': method,
            'path': path,
            'header_profile': None if str(ev.header_profile or 'none') == 'none' else str(ev.header_profile),
            'payload_class': None if str(ev.payload_class or 'none') == 'none' else str(ev.payload_class),
            'mutation_flags': mutation_flags,
            'expect_response_class': '2xx',
        })
    return events



## Training Entry Points

Train, eval, resume ve inference giris noktalari.


In [ ]:

def train_from_csv_dirs(*, train_csv_dir: str, val_csv_dir: str, out_dir: str, config: Dict[str, Any], device: str = 'cpu', csv_chunksize: int = 50000, max_train_rows_per_file: int = 0, max_val_rows_per_file: int = 0, max_train_sessions: int = 0, max_val_sessions: int = 0, num_workers: int = 0, batch_size_override: int = 0, epochs_override: int = 0, log_interval: int = 200) -> Dict[str, Any]:
    # Bu helper notebook icindeki ana egitim giris noktasi.
    # Ham CSV klasorlerinden session cikarir, vocab'i train setten kurar,
    # modeli egitir ve tek bir checkpoint + summary artefact uretir.
    train_sources = collect_csv_sources(csv_dir=str(train_csv_dir or '').strip(), csv_glob='*.csv', csv_files=[])
    val_sources = collect_csv_sources(csv_dir=str(val_csv_dir or '').strip(), csv_glob='*.csv', csv_files=[])
    if not train_sources:
        raise RuntimeError('No train CSV source found.')
    if not val_sources:
        raise RuntimeError('No val CSV source found.')

    train_sessions, train_rows_seen, train_files_seen = load_sessions_from_csv_sources(train_sources, max_sessions=int(max_train_sessions or 0), max_rows_per_file=int(max_train_rows_per_file or 0), csv_chunksize=int(csv_chunksize or 50000))
    val_sessions, val_rows_seen, val_files_seen = load_sessions_from_csv_sources(val_sources, max_sessions=int(max_val_sessions or 0), max_rows_per_file=int(max_val_rows_per_file or 0), csv_chunksize=int(csv_chunksize or 50000))
    if not train_sessions:
        raise RuntimeError('No train sessions extracted from CSV sources.')
    if not val_sessions:
        raise RuntimeError('No val sessions extracted from CSV sources.')

    train_label_set = {str(s.label or 'benign') for s in train_sessions}
    val_label_set = {str(s.label or 'benign') for s in val_sessions}
    missing_val_labels = sorted(val_label_set - train_label_set)
    if missing_val_labels:
        raise RuntimeError(f'Validation labels train vocab disinda kaliyor: {missing_val_labels}. Spliti veya train dosyalarini guncelleyin.')
    train_protocol_set = {str(s.protocol or 'http1') for s in train_sessions}
    val_protocol_set = {str(s.protocol or 'http1') for s in val_sessions}
    missing_val_protocols = sorted(val_protocol_set - train_protocol_set)
    if missing_val_protocols:
        raise RuntimeError(f'Validation protocol coverage train setinde yok: {missing_val_protocols}. Spliti veya train dosyalarini guncelleyin.')

    codec = TokenCodec.build(train_sessions)
    max_len = int((config.get('model', {}) or {}).get('max_sequence_length', 128) or 128)
    train_ds = SequenceDataset(train_sessions, codec, max_len=max_len)
    val_ds = SequenceDataset(val_sessions, codec, max_len=max_len)

    train_cfg = dict(config.get('training', {}) or {})
    seed = int(train_cfg.get('seed', 1337) or 1337)
    seed_everything(seed)
    torch_device = training_device(device)
    batch_size = int(batch_size_override or 0) if int(batch_size_override or 0) > 0 else int(train_cfg.get('batch_size', 64) or 64)
    epochs = int(epochs_override or 0) if int(epochs_override or 0) > 0 else int(train_cfg.get('epochs', 10) or 10)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=max(0, int(num_workers)), pin_memory=bool(torch_device.type == 'cuda'), drop_last=False)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=max(0, int(num_workers)), pin_memory=bool(torch_device.type == 'cuda'), drop_last=False)

    model = build_model_from_config(vocab_size=len(codec.vocab), config=config).to(torch_device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(train_cfg.get('learning_rate', 3e-4) or 3e-4), weight_decay=float(train_cfg.get('weight_decay', 0.01) or 0.01))

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    history_path = out_path / 'train_metrics.jsonl'
    best_val_ppl = float('inf')
    best_metrics: Dict[str, Any] = {}
    best_state_dict: Optional[Dict[str, Any]] = None
    best_optimizer_state: Optional[Dict[str, Any]] = None

    run_meta = {
        'status': 'starting',
        'device': str(torch_device),
        'seed': int(seed),
        'batch_size': int(batch_size),
        'epochs': int(epochs),
        'steps_per_epoch_train': int(len(train_loader)),
        'steps_per_epoch_val': int(len(val_loader)),
        'rows_train': int(train_rows_seen),
        'rows_val': int(val_rows_seen),
        'sessions_train': int(len(train_sessions)),
        'sessions_val': int(len(val_sessions)),
        'train_csv_files': [str(p) for _lbl, p in train_sources],
        'val_csv_files': [str(p) for _lbl, p in val_sources],
    }
    print(json.dumps(run_meta, sort_keys=True), flush=True)

    with history_path.open('w', encoding='utf-8') as history_f:
        for epoch in range(1, epochs + 1):
            train_metrics = run_epoch(model=model, loader=train_loader, optimizer=optimizer, device=torch_device, pad_id=codec.pad_id, epoch=int(epoch), log_interval=int(log_interval or 0))
            val_metrics = evaluate_model(model, val_loader, pad_id=codec.pad_id, device=torch_device)
            epoch_metrics = {'epoch': float(epoch), 'train_nll': float(train_metrics['nll']), 'val_acc': float(val_metrics['next_event_top1_accuracy']), 'val_ppl': float(val_metrics['transition_perplexity'])}
            history_f.write(json.dumps(epoch_metrics, sort_keys=True) + '\n')
            history_f.flush()
            print(json.dumps(epoch_metrics, sort_keys=True), flush=True)
            if float(val_metrics['transition_perplexity']) < best_val_ppl:
                best_val_ppl = float(val_metrics['transition_perplexity'])
                best_metrics = {
                    'best_epoch': int(epoch),
                    'best_val_acc': float(val_metrics['next_event_top1_accuracy']),
                    'best_val_ppl': float(val_metrics['transition_perplexity']),
                    'transitions_eval': int(val_metrics['transitions_eval']),
                }
                best_state_dict = copy.deepcopy(model.state_dict())
                best_optimizer_state = copy.deepcopy(optimizer.state_dict())

    if best_state_dict is None:
        best_state_dict = copy.deepcopy(model.state_dict())
        best_optimizer_state = copy.deepcopy(optimizer.state_dict())
        best_metrics = {
            'best_epoch': int(epochs),
            'best_val_acc': float(val_metrics['next_event_top1_accuracy']),
            'best_val_ppl': float(val_metrics['transition_perplexity']),
            'transitions_eval': int(val_metrics['transitions_eval']),
        }
    model.load_state_dict(best_state_dict, strict=True)
    if best_optimizer_state is not None:
        optimizer.load_state_dict(best_optimizer_state)

    ckpt_name = f"mamba2_v1_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.ckpt.pt"
    ckpt_path = out_path / ckpt_name
    save_checkpoint(str(ckpt_path), model=model, codec=codec, config=config, dataset_meta={
        'rows_train': int(train_rows_seen),
        'rows_val': int(val_rows_seen),
        'sessions_train': int(len(train_sessions)),
        'sessions_val': int(len(val_sessions)),
        'train_files_seen': int(train_files_seen),
        'val_files_seen': int(val_files_seen),
        'train_csv_files': [str(p) for _lbl, p in train_sources],
        'val_csv_files': [str(p) for _lbl, p in val_sources],
    }, metrics=best_metrics, optimizer=optimizer, train_state={'last_epoch': int(epochs), 'completed_epochs': int(epochs)})
    summary = {
        'status': 'ok',
        'config': config,
        'device': str(torch_device),
        'model_contract': 'multi_field_transition_lm_v2',
        'runtime_materialization': 'token_conditioned_v2',
        'rows_train': int(train_rows_seen),
        'rows_val': int(val_rows_seen),
        'sessions_train': int(len(train_sessions)),
        'sessions_val': int(len(val_sessions)),
        'checkpoint_path': str(ckpt_path),
        'checkpoint_sha1': hashlib.sha1(ckpt_path.read_bytes()).hexdigest(),
        'best_metrics': best_metrics,
        'metrics_history_path': str(history_path),
        'train_csv_files': [str(p) for _lbl, p in train_sources],
        'val_csv_files': [str(p) for _lbl, p in val_sources],
    }
    (out_path / 'train_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
    print(json.dumps(summary, indent=2, sort_keys=True))
    return summary


def eval_checkpoint_on_csv_dir(*, checkpoint_path: str, eval_csv_dir: str, device: str = 'cpu', batch_size: int = 64, num_workers: int = 0, csv_chunksize: int = 50000, max_eval_rows_per_file: int = 0, max_eval_sessions: int = 0) -> Dict[str, Any]:
    ckpt = Path(checkpoint_path)
    if not ckpt.exists():
        result = {'checkpoint_exists': False, 'checkpoint': str(ckpt), 'status': 'missing_checkpoint'}
        print(json.dumps(result, indent=2, sort_keys=True))
        return result
    eval_sources = collect_csv_sources(csv_dir=str(eval_csv_dir or '').strip(), csv_glob='*.csv', csv_files=[])
    if not eval_sources:
        raise RuntimeError('No eval CSV source found.')
    torch_device = training_device(device)
    model, codec, loaded = build_model_and_codec_from_checkpoint(str(ckpt), device=str(torch_device))
    sessions, rows_seen, files_seen = load_sessions_from_csv_sources(eval_sources, max_sessions=int(max_eval_sessions or 0), max_rows_per_file=int(max_eval_rows_per_file or 0), csv_chunksize=int(csv_chunksize or 50000))
    eval_labels = sorted({str(s.label or 'benign') for s in sessions})
    known_labels = {tok[len(LABEL_PREFIX):] for tok in codec.vocab if tok.startswith(LABEL_PREFIX)}
    missing_eval_labels = sorted(set(eval_labels) - known_labels)
    if missing_eval_labels:
        raise RuntimeError(f'Eval labels checkpoint vocab disinda kaliyor: {missing_eval_labels}. Train splitini veya checkpointi guncelleyin.')
    eval_protocols = sorted({str(s.protocol or 'http1') for s in sessions})
    known_protocols = {tok[len(PROTO_PREFIX):] for tok in codec.vocab if tok.startswith(PROTO_PREFIX)}
    missing_eval_protocols = sorted(set(eval_protocols) - known_protocols)
    if missing_eval_protocols:
        raise RuntimeError(f'Eval protocol coverage checkpoint vocab disinda kaliyor: {missing_eval_protocols}. Train splitini veya checkpointi guncelleyin.')
    max_len = int((loaded.get('config', {}) or {}).get('model', {}).get('max_sequence_length', 128) or 128)
    ds = SequenceDataset(sessions, codec, max_len=max_len)
    loader = torch.utils.data.DataLoader(ds, batch_size=max(1, int(batch_size)), shuffle=False, num_workers=max(0, int(num_workers)), pin_memory=bool(torch_device.type == 'cuda'), drop_last=False)
    metrics = evaluate_model(model, loader, pad_id=codec.pad_id, device=torch_device)
    result = {
        'checkpoint_exists': True,
        'checkpoint': str(ckpt),
        'status': 'ok',
        'model_contract': str(loaded.get('model_contract', 'multi_field_transition_lm_v2')),
        'runtime_materialization': str(loaded.get('runtime_materialization', 'token_conditioned_v2')),
        'eval_csv_files': [str(p) for _lbl, p in eval_sources],
        'eval_rows_seen': int(rows_seen),
        'eval_files_seen': int(files_seen),
        'sessions_eval': int(len(sessions)),
        'transitions_eval': int(metrics['transitions_eval']),
        'next_event_top1_accuracy': float(round(metrics['next_event_top1_accuracy'], 6)),
        'transition_perplexity': float(round(metrics['transition_perplexity'], 6)),
    }
    print(json.dumps(result, indent=2, sort_keys=True))
    return result


def find_latest_checkpoint(out_dir: str) -> Optional[str]:
    ckpts = sorted(Path(out_dir).glob('*.ckpt.pt'))
    if not ckpts:
        return None
    return str(ckpts[-1])


def resume_training_from_checkpoint(*, checkpoint_path: str, train_csv_dir: str, val_csv_dir: str, out_dir: str, additional_epochs: int, device: str = 'cpu', csv_chunksize: int = 50000, max_train_rows_per_file: int = 0, max_val_rows_per_file: int = 0, max_train_sessions: int = 0, max_val_sessions: int = 0, num_workers: int = 0, batch_size_override: int = 0, log_interval: int = 200) -> Dict[str, Any]:
    # Resume akisinda vocab ve model sekli checkpoint'ten gelir.
    # Bu kritik nokta su: resume sirasinda vocab'i tekrar train verisinden kurmuyoruz.
    # Aksi halde token id'leri kayar ve eski agirliklar anlamsiz hale gelir.
    ckpt = Path(checkpoint_path)
    if not ckpt.exists():
        raise RuntimeError(f'Checkpoint bulunamadi: {checkpoint_path}')
    loaded = load_checkpoint(str(ckpt), map_location='cpu' if str(device).lower() == 'cpu' else str(device))
    config = dict(loaded.get('config', {}) or {})
    train_sources = collect_csv_sources(csv_dir=str(train_csv_dir or '').strip(), csv_glob='*.csv', csv_files=[])
    val_sources = collect_csv_sources(csv_dir=str(val_csv_dir or '').strip(), csv_glob='*.csv', csv_files=[])
    if not train_sources:
        raise RuntimeError('No train CSV source found.')
    if not val_sources:
        raise RuntimeError('No val CSV source found.')

    train_sessions, train_rows_seen, train_files_seen = load_sessions_from_csv_sources(train_sources, max_sessions=int(max_train_sessions or 0), max_rows_per_file=int(max_train_rows_per_file or 0), csv_chunksize=int(csv_chunksize or 50000))
    val_sessions, val_rows_seen, val_files_seen = load_sessions_from_csv_sources(val_sources, max_sessions=int(max_val_sessions or 0), max_rows_per_file=int(max_val_rows_per_file or 0), csv_chunksize=int(csv_chunksize or 50000))
    if not train_sessions:
        raise RuntimeError('No train sessions extracted from CSV sources.')
    if not val_sessions:
        raise RuntimeError('No val sessions extracted from CSV sources.')

    train_label_set = {str(s.label or 'benign') for s in train_sessions}
    val_label_set = {str(s.label or 'benign') for s in val_sessions}
    missing_val_labels = sorted(val_label_set - train_label_set)
    if missing_val_labels:
        raise RuntimeError(f'Resume validation labels train vocab disinda kaliyor: {missing_val_labels}. Spliti veya train dosyalarini guncelleyin.')
    train_protocol_set = {str(s.protocol or 'http1') for s in train_sessions}
    val_protocol_set = {str(s.protocol or 'http1') for s in val_sessions}
    missing_val_protocols = sorted(val_protocol_set - train_protocol_set)
    if missing_val_protocols:
        raise RuntimeError(f'Resume validation protocol coverage train setinde yok: {missing_val_protocols}. Spliti veya train dosyalarini guncelleyin.')

    codec = TokenCodec(vocab=list(loaded['vocab']))
    max_len = int((config.get('model', {}) or {}).get('max_sequence_length', 128) or 128)
    train_ds = SequenceDataset(train_sessions, codec, max_len=max_len)
    val_ds = SequenceDataset(val_sessions, codec, max_len=max_len)

    train_cfg = dict(config.get('training', {}) or {})
    seed = int(train_cfg.get('seed', 1337) or 1337)
    seed_everything(seed)
    torch_device = training_device(device)
    batch_size = int(batch_size_override or 0) if int(batch_size_override or 0) > 0 else int(train_cfg.get('batch_size', 64) or 64)
    add_epochs = max(1, int(additional_epochs or 1))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=max(0, int(num_workers)), pin_memory=bool(torch_device.type == 'cuda'), drop_last=False)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=max(0, int(num_workers)), pin_memory=bool(torch_device.type == 'cuda'), drop_last=False)

    model = build_model_from_config(vocab_size=len(codec.vocab), config=config).to(torch_device)
    model.load_state_dict(dict(loaded.get('state_dict', {}) or {}), strict=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(train_cfg.get('learning_rate', 3e-4) or 3e-4), weight_decay=float(train_cfg.get('weight_decay', 0.01) or 0.01))
    if loaded.get('optimizer_state_dict'):
        optimizer.load_state_dict(dict(loaded.get('optimizer_state_dict', {}) or {}))

    previous_epochs = int((loaded.get('train_state', {}) or {}).get('completed_epochs', 0) or 0)
    history_path = Path(out_dir) / 'train_metrics_resume.jsonl'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    best_val_ppl = float('inf')
    best_metrics: Dict[str, Any] = {}
    best_state_dict: Optional[Dict[str, Any]] = None
    best_optimizer_state: Optional[Dict[str, Any]] = None

    with history_path.open('w', encoding='utf-8') as history_f:
        for local_epoch in range(1, add_epochs + 1):
            global_epoch = previous_epochs + local_epoch
            train_metrics = run_epoch(model=model, loader=train_loader, optimizer=optimizer, device=torch_device, pad_id=codec.pad_id, epoch=int(global_epoch), log_interval=int(log_interval or 0))
            val_metrics = evaluate_model(model, val_loader, pad_id=codec.pad_id, device=torch_device)
            epoch_metrics = {'epoch': float(global_epoch), 'train_nll': float(train_metrics['nll']), 'val_acc': float(val_metrics['next_event_top1_accuracy']), 'val_ppl': float(val_metrics['transition_perplexity'])}
            history_f.write(json.dumps(epoch_metrics, sort_keys=True) + '\n')
            history_f.flush()
            print(json.dumps(epoch_metrics, sort_keys=True), flush=True)
            if float(val_metrics['transition_perplexity']) < best_val_ppl:
                best_val_ppl = float(val_metrics['transition_perplexity'])
                best_metrics = {
                    'best_epoch': int(global_epoch),
                    'best_val_acc': float(val_metrics['next_event_top1_accuracy']),
                    'best_val_ppl': float(val_metrics['transition_perplexity']),
                    'transitions_eval': int(val_metrics['transitions_eval']),
                }
                best_state_dict = copy.deepcopy(model.state_dict())
                best_optimizer_state = copy.deepcopy(optimizer.state_dict())

    if best_state_dict is None:
        best_state_dict = copy.deepcopy(model.state_dict())
        best_optimizer_state = copy.deepcopy(optimizer.state_dict())
        best_metrics = {
            'best_epoch': int(previous_epochs + add_epochs),
            'best_val_acc': float(val_metrics['next_event_top1_accuracy']),
            'best_val_ppl': float(val_metrics['transition_perplexity']),
            'transitions_eval': int(val_metrics['transitions_eval']),
        }
    model.load_state_dict(best_state_dict, strict=True)
    if best_optimizer_state is not None:
        optimizer.load_state_dict(best_optimizer_state)

    resumed_epochs = previous_epochs + add_epochs
    ckpt_name = f"mamba2_resume_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.ckpt.pt"
    ckpt_path = Path(out_dir) / ckpt_name
    save_checkpoint(str(ckpt_path), model=model, codec=codec, config=config, dataset_meta={
        'rows_train': int(train_rows_seen),
        'rows_val': int(val_rows_seen),
        'sessions_train': int(len(train_sessions)),
        'sessions_val': int(len(val_sessions)),
        'train_files_seen': int(train_files_seen),
        'val_files_seen': int(val_files_seen),
        'train_csv_files': [str(p) for _lbl, p in train_sources],
        'val_csv_files': [str(p) for _lbl, p in val_sources],
        'resumed_from': str(ckpt),
    }, metrics=best_metrics, optimizer=optimizer, train_state={'last_epoch': int(resumed_epochs), 'completed_epochs': int(resumed_epochs), 'resumed_from': str(ckpt)})
    summary = {
        'status': 'ok',
        'mode': 'resume',
        'config': config,
        'device': str(torch_device),
        'checkpoint_path': str(ckpt_path),
        'checkpoint_sha1': hashlib.sha1(ckpt_path.read_bytes()).hexdigest(),
        'resumed_from': str(ckpt),
        'previous_completed_epochs': int(previous_epochs),
        'additional_epochs': int(add_epochs),
        'completed_epochs': int(resumed_epochs),
        'best_metrics': best_metrics,
        'metrics_history_path': str(history_path),
    }
    (Path(out_dir) / 'resume_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
    print(json.dumps(summary, indent=2, sort_keys=True))
    return summary


def infer_plan_from_checkpoint(*, checkpoint_path: str, protocol: str, label: str, session_count: int, session_len: int, seed: int, device: str = 'cpu', temperature: float = 0.8, top_k: int = 20) -> TrafficPlan:
    # Inference iki asamalidir:
    # 1. Model token seviyesinde cok alanli EventRecord dizisi uretir.
    # 2. Materializer bu soyut alanlari runtime-benzeri event objelerine acar.
    model, codec, loaded = build_model_and_codec_from_checkpoint(checkpoint_path, device=device)
    session_plans: List[SessionPlan] = []
    for i in range(int(session_count)):
        evs = sample_event_types(model, codec, protocol=protocol, label=label, max_events=max(1, int(session_len) - 1), seed=int(seed) + i, device=device, temperature=float(temperature), top_k=int(top_k))
        materialized = materialize_session_events(evs, protocol=protocol, label=label, session_len=int(session_len), seed=int(seed) + i)
        session_plans.append(SessionPlan(
            session_id=f'infer-{protocol}-{label}-{i:04d}',
            protocol=str(protocol),
            label=str(label),
            events=[SessionEvent(**ev) for ev in materialized],
        ))
    plan = TrafficPlan(
        plan_id=f'plan-{seed}',
        model_name=str(loaded.get('model_name', 'mamba2-small')),
        checkpoint_id=hashlib.sha1(Path(checkpoint_path).read_bytes()).hexdigest()[:12],
        seed=int(seed),
        session_plans=session_plans,
    )
    vr = validate_plan(plan)
    if not vr.valid:
        raise RuntimeError(f'Inference plan validation failed: {vr.errors}')
    return plan


In [ ]:
# Yerel calisma ayarlari
from pathlib import Path

def _find_code_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'training' / 'mamba2_monolith.ipynb').exists() and (candidate / 'ddos_samurai').exists():
            return candidate
    raise RuntimeError('CODE_ROOT otomatik bulunamadi. Notebooku repo icinden calistirin veya ortam degiskeni verin.')

CODE_ROOT = Path(os.environ.get('MAMBA_MONOLITH_CODE_ROOT', str(_find_code_root()))).resolve()
DATA_ROOT = Path(os.environ.get('MAMBA_MONOLITH_DATA_ROOT', str(CODE_ROOT / 'data' / 'raw' / 'cicids2017'))).resolve()
WORK_ROOT = Path(os.environ.get('MAMBA_MONOLITH_WORK_ROOT', str(CODE_ROOT / 'notebook_runs' / 'mamba2_monolith'))).resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.exists():
    raise RuntimeError(f'DATA_ROOT bulunamadi: {DATA_ROOT}')

CSV_FILES = sorted(DATA_ROOT.glob('*.csv'))
print('CODE_ROOT =', CODE_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('WORK_ROOT =', WORK_ROOT)
print('CSV count =', len(CSV_FILES))
for p in CSV_FILES:
    print(' -', p.name)


In [ ]:
MANUAL_TRAIN_FILES: List[str] = []
MANUAL_VAL_FILES: List[str] = []
MANUAL_TEST_FILES: List[str] = []
AUTO_TEST_KEYWORDS = ('ddos', 'portscan')

split_root = WORK_ROOT / 'csv_split'
for part in ('train', 'val', 'test'):
    (split_root / part).mkdir(parents=True, exist_ok=True)

available = {p.name: p for p in CSV_FILES}


def _read_label_values(csv_path: Path) -> List[str]:
    for col in (' Label', 'Label', 'label'):
        try:
            df = pd.read_csv(csv_path, usecols=[col], low_memory=False)
            return sorted({str(x).strip() for x in df.iloc[:, 0].dropna().unique().tolist() if str(x).strip()})
        except Exception:
            continue
    return []


def _file_label_profiles(files: Sequence[Path]) -> Dict[str, List[str]]:
    profiles: Dict[str, List[str]] = {}
    for csv_path in files:
        mapped = sorted({_map_label(raw) for raw in _read_label_values(csv_path)})
        profiles[csv_path.name] = mapped
    return profiles


def _coverage(file_names: Sequence[str], profiles: Dict[str, List[str]]) -> set[str]:
    covered: set[str] = set()
    for name in file_names:
        covered.update(profiles.get(name, []))
    return covered


def _choose_auto_split(files: Sequence[Path]) -> tuple[List[str], List[str], List[str], Dict[str, List[str]]]:
    profiles = _file_label_profiles(files)
    names = sorted(profiles)
    test_files = sorted([name for name in names if any(key in name.lower() for key in AUTO_TEST_KEYWORDS)])
    remaining = [name for name in names if name not in test_files]
    if not remaining:
        raise RuntimeError('Auto split icin train/val adayi kalmadi.')

    candidates: List[tuple[tuple[int, int, int, int, str], str]] = []
    for name in remaining:
        train_files = [candidate for candidate in remaining if candidate != name]
        if not train_files:
            continue
        train_cov = _coverage(train_files, profiles)
        target_cov = _coverage([name] + test_files, profiles)
        if not target_cov.issubset(train_cov):
            continue
        labels = set(profiles.get(name, []))
        score = (
            int('evasive' in labels),
            int('benign' in labels),
            -int('aggressive' in labels),
            -len(labels),
            name,
        )
        candidates.append((score, name))
    if not candidates:
        raise RuntimeError('Auto split train coverage koruyarak val secemedi. Manuel override kullanin.')

    candidates.sort(reverse=True)
    val_files = [candidates[0][1]]
    train_files = [name for name in remaining if name not in val_files]
    return sorted(train_files), sorted(val_files), sorted(test_files), profiles


if MANUAL_TRAIN_FILES or MANUAL_VAL_FILES or MANUAL_TEST_FILES:
    TRAIN_FILES = sorted(MANUAL_TRAIN_FILES)
    VAL_FILES = sorted(MANUAL_VAL_FILES)
    TEST_FILES = sorted(MANUAL_TEST_FILES)
    file_profiles = _file_label_profiles(CSV_FILES)
    split_strategy = 'manual'
else:
    TRAIN_FILES, VAL_FILES, TEST_FILES, file_profiles = _choose_auto_split(CSV_FILES)
    split_strategy = 'auto_label_coverage'

missing = sorted((set(TRAIN_FILES) | set(VAL_FILES) | set(TEST_FILES)) - set(available))
if missing:
    raise RuntimeError(f'Splitte eksik dosyalar var: {missing}')
if set(TRAIN_FILES) & set(VAL_FILES):
    raise RuntimeError('Train ve val dosya listeleri cakismamali.')
if set(TRAIN_FILES) & set(TEST_FILES):
    raise RuntimeError('Train ve test dosya listeleri cakismamali.')
if set(VAL_FILES) & set(TEST_FILES):
    raise RuntimeError('Val ve test dosya listeleri cakismamali.')
if not TRAIN_FILES or not VAL_FILES or not TEST_FILES:
    raise RuntimeError('Train/val/test bos birakilamaz.')

for part, names in [('train', TRAIN_FILES), ('val', VAL_FILES), ('test', TEST_FILES)]:
    out_dir = split_root / part
    for old in out_dir.glob('*.csv'):
        old.unlink()
    for name in names:
        shutil.copy2(available[name], out_dir / name)

for part in ('train', 'val', 'test'):
    print(part, sorted(p.name for p in (split_root / part).glob('*.csv')))

split_labels = {}
for part in ('train', 'val', 'test'):
    labels = set()
    for csv_path in sorted((split_root / part).glob('*.csv')):
        labels.update(_map_label(raw) for raw in _read_label_values(csv_path))
    split_labels[part] = sorted(labels)

print('split_strategy =', split_strategy)
print('file_label_profiles =', json.dumps(file_profiles, indent=2, sort_keys=True))
print('split_labels =', split_labels)

missing_val_labels = sorted(set(split_labels['val']) - set(split_labels['train']))
missing_test_labels = sorted(set(split_labels['test']) - set(split_labels['train']))
if missing_val_labels:
    raise RuntimeError(f'Val labels train coverage disinda: {missing_val_labels}')
if missing_test_labels:
    raise RuntimeError(f'Test labels train coverage disinda: {missing_test_labels}')


In [ ]:
PROFILES = {
    'gtx1650_4gb': {
        'model': {
            'name': 'mamba2-monolith-1650-4gb',
            'max_sequence_length': 96,
            'hidden_size': 256,
            'num_layers': 6,
            'd_state': 16,
            'd_conv': 4,
            'expand': 2,
        },
        'training': {
            'epochs': 3,
            'batch_size': 2,
            'learning_rate': 3e-4,
            'weight_decay': 0.01,
            'seed': 1337,
        },
    },
    'ultra_low_vram': {
        'model': {
            'name': 'mamba2-monolith-ultra-low-vram',
            'max_sequence_length': 64,
            'hidden_size': 192,
            'num_layers': 4,
            'd_state': 16,
            'd_conv': 4,
            'expand': 2,
        },
        'training': {
            'epochs': 3,
            'batch_size': 1,
            'learning_rate': 3e-4,
            'weight_decay': 0.01,
            'seed': 1337,
        },
    },
}

SMOKE_MODE = str(os.environ.get('MAMBA_MONOLITH_SMOKE', '0')).strip().lower() in {'1', 'true', 'yes', 'on'}
PROFILE_NAME = os.environ.get('MAMBA_MONOLITH_PROFILE', 'gtx1650_4gb').strip() or 'gtx1650_4gb'
if PROFILE_NAME not in PROFILES:
    raise RuntimeError(f'Bilinmeyen PROFILE_NAME: {PROFILE_NAME}')

CONFIG = json.loads(json.dumps(PROFILES[PROFILE_NAME]))
CONFIG['objective'] = {
    'type': 'autoregressive_multi_field_sequence',
    'fields': [
        'event_type',
        'delay_bucket',
        'method',
        'path_bucket',
        'header_profile',
        'payload_class',
        'mutation_signature',
    ],
}

if SMOKE_MODE:
    CONFIG['model'].update({
        'name': f"{CONFIG['model']['name']}-smoke",
        'max_sequence_length': 48,
        'hidden_size': 64,
        'num_layers': 2,
        'd_state': 8,
        'd_conv': 4,
        'expand': 2,
    })
    CONFIG['training'].update({
        'epochs': 1,
        'batch_size': 1,
        'learning_rate': 3e-4,
        'weight_decay': 0.01,
        'seed': 1337,
    })

DEVICE = os.environ.get('MAMBA_MONOLITH_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu').strip().lower() or 'cpu'
CSV_CHUNKSIZE = 1000 if SMOKE_MODE else 10000
TRAIN_RUN_LIMITS = {
    'max_train_rows_per_file': 500 if SMOKE_MODE else 0,
    'max_val_rows_per_file': 300 if SMOKE_MODE else 0,
    'max_train_sessions': 64 if SMOKE_MODE else 0,
    'max_val_sessions': 32 if SMOKE_MODE else 0,
}
EVAL_RUN_LIMITS = {
    'max_eval_rows_per_file': 300 if SMOKE_MODE else 0,
    'max_eval_sessions': 32 if SMOKE_MODE else 0,
}
PLAN_SETTINGS = {
    'protocol': 'http1',
    'label': 'benign',
    'session_count': 1 if SMOKE_MODE else 2,
    'session_len': 6,
    'seed': 1337,
    'temperature': 0.8,
    'top_k': 10,
}
RESUME_EXTRA_EPOCHS = 1 if SMOKE_MODE else 2
OUT_DIR_NAME = 'smoke_run' if SMOKE_MODE else 'run1'
OUT_DIR = WORK_ROOT / OUT_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('SMOKE_MODE =', SMOKE_MODE)
print('PROFILE_NAME =', PROFILE_NAME)
print(json.dumps(CONFIG, indent=2))
print('DEVICE =', DEVICE)
print('CSV_CHUNKSIZE =', CSV_CHUNKSIZE)
print('TRAIN_RUN_LIMITS =', TRAIN_RUN_LIMITS)
print('EVAL_RUN_LIMITS =', EVAL_RUN_LIMITS)
print('RESUME_EXTRA_EPOCHS =', RESUME_EXTRA_EPOCHS)
print('OUT_DIR =', OUT_DIR)


In [ ]:
train_summary = train_from_csv_dirs(
    train_csv_dir=str(split_root / 'train'),
    val_csv_dir=str(split_root / 'val'),
    out_dir=str(OUT_DIR),
    config=CONFIG,
    device=DEVICE,
    csv_chunksize=CSV_CHUNKSIZE,
    batch_size_override=int(CONFIG['training']['batch_size']),
    epochs_override=int(CONFIG['training']['epochs']),
    log_interval=20 if not SMOKE_MODE else 0,
    **TRAIN_RUN_LIMITS,
)
CHECKPOINT_PATH = train_summary['checkpoint_path']
print('CHECKPOINT_PATH =', CHECKPOINT_PATH)


In [ ]:
RESUME_FROM = find_latest_checkpoint(str(OUT_DIR)) or CHECKPOINT_PATH
print('RESUME_FROM =', RESUME_FROM)

resume_summary = resume_training_from_checkpoint(
    checkpoint_path=str(RESUME_FROM),
    train_csv_dir=str(split_root / 'train'),
    val_csv_dir=str(split_root / 'val'),
    out_dir=str(OUT_DIR),
    additional_epochs=int(RESUME_EXTRA_EPOCHS),
    device=DEVICE,
    csv_chunksize=CSV_CHUNKSIZE,
    batch_size_override=int(CONFIG['training']['batch_size']),
    log_interval=20 if not SMOKE_MODE else 0,
    max_train_rows_per_file=int(TRAIN_RUN_LIMITS['max_train_rows_per_file']),
    max_val_rows_per_file=int(TRAIN_RUN_LIMITS['max_val_rows_per_file']),
    max_train_sessions=int(TRAIN_RUN_LIMITS['max_train_sessions']),
    max_val_sessions=int(TRAIN_RUN_LIMITS['max_val_sessions']),
)
CHECKPOINT_PATH = resume_summary['checkpoint_path']
print('RESUMED CHECKPOINT_PATH =', CHECKPOINT_PATH)


In [ ]:
eval_result = eval_checkpoint_on_csv_dir(
    checkpoint_path=str(CHECKPOINT_PATH),
    eval_csv_dir=str(split_root / 'test'),
    device=DEVICE,
    batch_size=2 if SMOKE_MODE else 4,
    csv_chunksize=CSV_CHUNKSIZE,
    **EVAL_RUN_LIMITS,
)


In [ ]:
plan = infer_plan_from_checkpoint(
    checkpoint_path=str(CHECKPOINT_PATH),
    protocol=PLAN_SETTINGS['protocol'],
    label=PLAN_SETTINGS['label'],
    session_count=int(PLAN_SETTINGS['session_count']),
    session_len=int(PLAN_SETTINGS['session_len']),
    seed=int(PLAN_SETTINGS['seed']),
    device=DEVICE,
    temperature=float(PLAN_SETTINGS['temperature']),
    top_k=int(PLAN_SETTINGS['top_k']),
)
print(json.dumps(plan.to_dict(), indent=2))
